In [ ]:
%pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

In [ ]:
spark = SparkSession.builder.appName("twitter_transformation").getOrCreate()

In [ ]:
df = spark.read.json("../datalake/Bronze/twitter_datascience/*/*.json")
df.show()

In [ ]:
df.printSchema()

In [ ]:
tweet_df = df.select(f.explode("data").alias("tweets"))\
    .select(
        "tweets.author_id",
        "tweets.conversation_id",
        "tweets.created_at",
        "tweets.id",
        "tweets.public_metrics.*", # wildcard: all fields of the struct
        "tweets.text")
    
tweet_df.show(3)

In [ ]:
df.printSchema()

In [ ]:
user_df = df.select(f.explode("includes.users").alias("users")).select("users.*")
user_df.show(3)

In [ ]:
# Save the tables as JSON
tweet_df.coalesce(1).write.mode("overwrite").json("output/tweet")
user_df.coalesce(1).write.mode("overwrite").json("output/user")